# Prototype Readiness Test

This notebook verifies that the saved Random Forest pipeline can load the
processed road dataset and generate a valid next-hour probe-count prediction
before integration with the Streamlit prototype.

In [1]:
from google.colab import files

print("Upload the following files:")
print("1. random_forest_next_hour_traffic_model.joblib")
print("2. selected_delhi_roads_hourly.csv")

uploaded = files.upload()

Upload the following files:
1. random_forest_next_hour_traffic_model.joblib
2. selected_delhi_roads_hourly.csv


Saving random_forest_next_hour_traffic_model.joblib to random_forest_next_hour_traffic_model.joblib


In [2]:
from google.colab import files

uploaded_dataset = files.upload()

Saving selected_delhi_roads_hourly.csv to selected_delhi_roads_hourly.csv


In [3]:
import os

required_files = [
    "random_forest_next_hour_traffic_model.joblib",
    "selected_delhi_roads_hourly.csv"
]

for filename in required_files:
    print(
        "[FOUND]" if os.path.exists(filename) else "[MISSING]",
        filename
    )

[FOUND] random_forest_next_hour_traffic_model.joblib
[FOUND] selected_delhi_roads_hourly.csv


In [4]:
import joblib
import numpy as np
import pandas as pd

MODEL_FILE = (
    "random_forest_next_hour_traffic_model.joblib"
)

DATASET_FILE = (
    "selected_delhi_roads_hourly.csv"
)

random_forest_pipeline = joblib.load(
    MODEL_FILE
)

traffic_df = pd.read_csv(
    DATASET_FILE
)

print("MODEL AND DATASET CHECK")
print("=" * 70)

print(
    "Model type:",
    type(random_forest_pipeline).__name__
)

print(
    "Pipeline steps:",
    list(
        random_forest_pipeline.named_steps.keys()
    )
)

print(
    "Dataset shape:",
    traffic_df.shape
)

print(
    "Number of roads:",
    traffic_df["street_name"].nunique()
)

print(
    "Date range:",
    traffic_df["date"].min(),
    "to",
    traffic_df["date"].max()
)

print("\nRoads available:")

for road in sorted(
    traffic_df["street_name"].unique()
):
    print("-", road)

MODEL AND DATASET CHECK
Model type: Pipeline
Pipeline steps: ['preprocessor', 'model']
Dataset shape: (4800, 12)
Number of roads: 10
Date range: 2024-08-11 to 2024-08-30

Roads available:
- Africa Avenue
- Aurobindo Marg
- Barapullah Road
- Mahatma Gandhi Marg
- Mathura Road
- Mehrauli Badarpur Road
- Noida Link Road
- Outer Ring Road
- Sardar Patel Marg
- Vikas Marg


In [5]:
prototype_df = traffic_df.copy()

# Combine the date and hour into a complete timestamp.
prototype_df["datetime"] = (
    pd.to_datetime(prototype_df["date"])
    + pd.to_timedelta(
        prototype_df["hour"],
        unit="h"
    )
)

# Keep every road in chronological order.
prototype_df = (
    prototype_df
    .sort_values(
        ["street_name", "datetime"]
    )
    .reset_index(drop=True)
)

road_groups = prototype_df.groupby(
    "street_name",
    group_keys=False
)

# Historical traffic features.
prototype_df["lag_1"] = (
    road_groups["probe_count"]
    .shift(1)
)

prototype_df["lag_24"] = (
    road_groups["probe_count"]
    .shift(24)
)

prototype_df["rolling_mean_3"] = (
    road_groups["probe_count"]
    .transform(
        lambda series:
        series.shift(1).rolling(3).mean()
    )
)

prototype_df["rolling_mean_24"] = (
    road_groups["probe_count"]
    .transform(
        lambda series:
        series.shift(1).rolling(24).mean()
    )
)

# Create the next-hour prediction target.
prototype_df["target_next_hour"] = (
    road_groups["probe_count"]
    .shift(-1)
)

prototype_df["target_datetime"] = (
    prototype_df["datetime"]
    + pd.Timedelta(hours=1)
)

prototype_df["target_hour"] = (
    prototype_df["target_datetime"].dt.hour
)

prototype_df["target_day_number"] = (
    prototype_df["target_datetime"].dt.dayofweek
)

prototype_df["target_is_weekend"] = (
    prototype_df["target_day_number"] >= 5
).astype(int)

prototype_df["target_is_peak_hour"] = (
    prototype_df["target_hour"]
    .isin(
        [7, 8, 9, 17, 18, 19, 20]
    )
    .astype(int)
)

# Remove records that do not yet have sufficient history.
prototype_model_df = (
    prototype_df
    .dropna(
        subset=[
            "lag_1",
            "lag_24",
            "rolling_mean_3",
            "rolling_mean_24",
            "target_next_hour"
        ]
    )
    .copy()
)

print("PROTOTYPE DATA CHECK")
print("=" * 70)

print(
    "Model-ready records:",
    len(prototype_model_df)
)

print(
    "Number of roads:",
    prototype_model_df[
        "street_name"
    ].nunique()
)

print(
    "First target:",
    prototype_model_df[
        "target_datetime"
    ].min()
)

print(
    "Last target:",
    prototype_model_df[
        "target_datetime"
    ].max()
)

print(
    "Missing values in model-ready data:",
    prototype_model_df[
        [
            "lag_1",
            "lag_24",
            "rolling_mean_3",
            "rolling_mean_24",
            "target_next_hour"
        ]
    ].isna().sum().sum()
)

PROTOTYPE DATA CHECK
Model-ready records: 4550
Number of roads: 10
First target: 2024-08-12 01:00:00
Last target: 2024-08-30 23:00:00
Missing values in model-ready data: 0


In [6]:
prototype_df = traffic_df.copy()

# Combine the date and hour into a complete timestamp.
prototype_df["datetime"] = (
    pd.to_datetime(prototype_df["date"])
    + pd.to_timedelta(
        prototype_df["hour"],
        unit="h"
    )
)

# Keep every road in chronological order.
prototype_df = (
    prototype_df
    .sort_values(
        ["street_name", "datetime"]
    )
    .reset_index(drop=True)
)

road_groups = prototype_df.groupby(
    "street_name",
    group_keys=False
)

# Historical traffic features.
prototype_df["lag_1"] = (
    road_groups["probe_count"]
    .shift(1)
)

prototype_df["lag_24"] = (
    road_groups["probe_count"]
    .shift(24)
)

prototype_df["rolling_mean_3"] = (
    road_groups["probe_count"]
    .transform(
        lambda series:
        series.shift(1).rolling(3).mean()
    )
)

prototype_df["rolling_mean_24"] = (
    road_groups["probe_count"]
    .transform(
        lambda series:
        series.shift(1).rolling(24).mean()
    )
)

# Create the next-hour prediction target.
prototype_df["target_next_hour"] = (
    road_groups["probe_count"]
    .shift(-1)
)

prototype_df["target_datetime"] = (
    prototype_df["datetime"]
    + pd.Timedelta(hours=1)
)

prototype_df["target_hour"] = (
    prototype_df["target_datetime"].dt.hour
)

prototype_df["target_day_number"] = (
    prototype_df["target_datetime"].dt.dayofweek
)

prototype_df["target_is_weekend"] = (
    prototype_df["target_day_number"] >= 5
).astype(int)

prototype_df["target_is_peak_hour"] = (
    prototype_df["target_hour"]
    .isin(
        [7, 8, 9, 17, 18, 19, 20]
    )
    .astype(int)
)

# Remove records that do not yet have sufficient history.
prototype_model_df = (
    prototype_df
    .dropna(
        subset=[
            "lag_1",
            "lag_24",
            "rolling_mean_3",
            "rolling_mean_24",
            "target_next_hour"
        ]
    )
    .copy()
)

print("PROTOTYPE DATA CHECK")
print("=" * 70)

print(
    "Model-ready records:",
    len(prototype_model_df)
)

print(
    "Number of roads:",
    prototype_model_df[
        "street_name"
    ].nunique()
)

print(
    "First target:",
    prototype_model_df[
        "target_datetime"
    ].min()
)

print(
    "Last target:",
    prototype_model_df[
        "target_datetime"
    ].max()
)

print(
    "Missing values in model-ready data:",
    prototype_model_df[
        [
            "lag_1",
            "lag_24",
            "rolling_mean_3",
            "rolling_mean_24",
            "target_next_hour"
        ]
    ].isna().sum().sum()
)

PROTOTYPE DATA CHECK
Model-ready records: 4550
Number of roads: 10
First target: 2024-08-12 01:00:00
Last target: 2024-08-30 23:00:00
Missing values in model-ready data: 0


In [7]:
feature_columns = [
    "street_name",
    "probe_count",
    "lag_1",
    "lag_24",
    "rolling_mean_3",
    "rolling_mean_24",
    "target_hour",
    "target_day_number",
    "target_is_weekend",
    "target_is_peak_hour",
    "segment_count",
    "average_speed_limit",
    "average_frc",
    "total_distance"
]

# Use the latest model-ready Mathura Road observation.
sample_road = "Mathura Road"

sample_rows = prototype_model_df[
    prototype_model_df["street_name"] == sample_road
].copy()

if sample_rows.empty:
    raise ValueError(
        f"No model-ready observations were found for {sample_road}."
    )

sample_row = sample_rows.iloc[-1]

sample_input = pd.DataFrame(
    [sample_row[feature_columns]]
)

predicted_probe_count = random_forest_pipeline.predict(
    sample_input
)[0]

actual_probe_count = sample_row["target_next_hour"]

absolute_error = abs(
    actual_probe_count - predicted_probe_count
)

percentage_error = (
    absolute_error / actual_probe_count * 100
    if actual_probe_count != 0
    else np.nan
)

print("PROTOTYPE PREDICTION TEST")
print("=" * 70)

print(
    "Road:",
    sample_row["street_name"]
)

print(
    "Current observation time:",
    sample_row["datetime"]
)

print(
    "Prediction target time:",
    sample_row["target_datetime"]
)

print(
    "Current probe count:",
    f'{sample_row["probe_count"]:,.2f}'
)

print(
    "Predicted next-hour probe count:",
    f"{predicted_probe_count:,.2f}"
)

print(
    "Actual next-hour probe count:",
    f"{actual_probe_count:,.2f}"
)

print(
    "Absolute error:",
    f"{absolute_error:,.2f}"
)

print(
    "Percentage error:",
    f"{percentage_error:.2f}%"
)

print(
    "\nPrediction generated successfully:",
    np.isfinite(predicted_probe_count)
)

PROTOTYPE PREDICTION TEST
Road: Mathura Road
Current observation time: 2024-08-30 22:00:00
Prediction target time: 2024-08-30 23:00:00
Current probe count: 165,549.00
Predicted next-hour probe count: 136,529.11
Actual next-hour probe count: 137,616.00
Absolute error: 1,086.89
Percentage error: 0.79%

Prediction generated successfully: True


In [8]:
print("MODEL INPUT USED FOR THE PREDICTION")
print("=" * 70)

display(
    sample_input.T.rename(
        columns={sample_input.index[0]: "Value"}
    )
)

MODEL INPUT USED FOR THE PREDICTION


,Value
street_name,Mathura Road
probe_count,165549
lag_1,197682.0
lag_24,138084.0
rolling_mean_3,200099.333333
rolling_mean_24,146984.416667
target_hour,23
target_day_number,4
target_is_weekend,0
target_is_peak_hour,0


In [9]:
prototype_test_result = pd.DataFrame({
    "Test_ID": ["PT-01"],
    "Road": [sample_row["street_name"]],
    "Current_Observation_Time": [
        sample_row["datetime"]
    ],
    "Prediction_Target_Time": [
        sample_row["target_datetime"]
    ],
    "Current_Probe_Count": [
        sample_row["probe_count"]
    ],
    "Predicted_Next_Hour_Probe_Count": [
        predicted_probe_count
    ],
    "Actual_Next_Hour_Probe_Count": [
        actual_probe_count
    ],
    "Absolute_Error": [
        absolute_error
    ],
    "Percentage_Error": [
        percentage_error
    ],
    "Prediction_Valid": [
        bool(np.isfinite(predicted_probe_count))
    ]
})

display(
    prototype_test_result.round(4)
)

prototype_test_result.to_csv(
    "prototype_readiness_test_result.csv",
    index=False
)

print(
    "Saved: prototype_readiness_test_result.csv"
)

,Test_ID,Road,Current_Observation_Time,Prediction_Target_Time,Current_Probe_Count,Predicted_Next_Hour_Probe_Count,Actual_Next_Hour_Probe_Count,Absolute_Error,Percentage_Error,Prediction_Valid
0,PT-01,Mathura Road,2024-08-30 22:00:00,2024-08-30 23:00:00,165549,136529.1054,137616.0,1086.8946,0.7898,True


Saved: prototype_readiness_test_result.csv


In [10]:
%%writefile app.py

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import streamlit as st


# ---------------------------------------------------------
# Application configuration
# ---------------------------------------------------------

st.set_page_config(
    page_title=(
        "New Delhi Traffic Probe Count Forecasting"
    ),
    page_icon="🚦",
    layout="wide"
)


BASE_DIRECTORY = Path(__file__).resolve().parent

MODEL_PATH = (
    BASE_DIRECTORY
    / "random_forest_next_hour_traffic_model.joblib"
)

DATASET_PATH = (
    BASE_DIRECTORY
    / "selected_delhi_roads_hourly.csv"
)


FEATURE_COLUMNS = [
    "street_name",
    "probe_count",
    "lag_1",
    "lag_24",
    "rolling_mean_3",
    "rolling_mean_24",
    "target_hour",
    "target_day_number",
    "target_is_weekend",
    "target_is_peak_hour",
    "segment_count",
    "average_speed_limit",
    "average_frc",
    "total_distance"
]


PEAK_HOURS = [
    7,
    8,
    9,
    17,
    18,
    19,
    20
]


# ---------------------------------------------------------
# Loading functions
# ---------------------------------------------------------

@st.cache_resource
def load_model(model_path):
    """Load the saved Random Forest pipeline."""

    if not model_path.exists():
        raise FileNotFoundError(
            f"Model file was not found: {model_path.name}"
        )

    return joblib.load(model_path)


@st.cache_data
def prepare_dataset(dataset_path):
    """
    Load the processed dataset and recreate the historical
    forecasting features required by the trained model.
    """

    if not dataset_path.exists():
        raise FileNotFoundError(
            f"Dataset file was not found: {dataset_path.name}"
        )

    data = pd.read_csv(dataset_path)

    data["datetime"] = (
        pd.to_datetime(data["date"])
        + pd.to_timedelta(
            data["hour"],
            unit="h"
        )
    )

    data = (
        data
        .sort_values(
            ["street_name", "datetime"]
        )
        .reset_index(drop=True)
    )

    grouped_data = data.groupby(
        "street_name",
        group_keys=False
    )

    data["lag_1"] = (
        grouped_data["probe_count"]
        .shift(1)
    )

    data["lag_24"] = (
        grouped_data["probe_count"]
        .shift(24)
    )

    data["rolling_mean_3"] = (
        grouped_data["probe_count"]
        .transform(
            lambda series:
            series.shift(1).rolling(3).mean()
        )
    )

    data["rolling_mean_24"] = (
        grouped_data["probe_count"]
        .transform(
            lambda series:
            series.shift(1).rolling(24).mean()
        )
    )

    data["target_next_hour"] = (
        grouped_data["probe_count"]
        .shift(-1)
    )

    data["target_datetime"] = (
        data["datetime"]
        + pd.Timedelta(hours=1)
    )

    data["target_hour"] = (
        data["target_datetime"].dt.hour
    )

    data["target_day_number"] = (
        data["target_datetime"].dt.dayofweek
    )

    data["target_is_weekend"] = (
        data["target_day_number"] >= 5
    ).astype(int)

    data["target_is_peak_hour"] = (
        data["target_hour"]
        .isin(PEAK_HOURS)
        .astype(int)
    )

    model_ready_data = (
        data
        .dropna(
            subset=[
                "lag_1",
                "lag_24",
                "rolling_mean_3",
                "rolling_mean_24",
                "target_next_hour"
            ]
        )
        .copy()
    )

    model_ready_data[
        "observation_date"
    ] = (
        model_ready_data[
            "datetime"
        ].dt.date
    )

    return model_ready_data


# ---------------------------------------------------------
# Load project resources
# ---------------------------------------------------------

try:
    forecasting_model = load_model(
        MODEL_PATH
    )

    forecasting_data = prepare_dataset(
        DATASET_PATH
    )

except Exception as error:
    st.error(
        "The application could not load its required "
        "project resources."
    )

    st.exception(error)
    st.stop()


# ---------------------------------------------------------
# Page heading
# ---------------------------------------------------------

st.title(
    "New Delhi Next-Hour Traffic Probe Count Forecasting"
)

st.caption(
    "Random Forest forecasting prototype for ten selected "
    "New Delhi roads"
)

st.info(
    "This prototype operates in historical evaluation mode. "
    "It uses an observation from the processed dataset to "
    "forecast the following hour and compare the prediction "
    "with the recorded historical value."
)


# ---------------------------------------------------------
# Sidebar
# ---------------------------------------------------------

with st.sidebar:

    st.header("Project Information")

    st.write(
        "**Student:** Suyambu Raj"
    )

    st.write(
        "**Model:** Random Forest Regressor"
    )

    st.write(
        "**Forecast horizon:** One hour"
    )

    st.write(
        "**Dataset coverage:** 11–30 August 2024"
    )

    st.write(
        "**Selected roads:** 10"
    )

    st.divider()

    st.warning(
        "Probe count represents aggregated probe activity. "
        "It is not an exact vehicle count or a direct "
        "congestion percentage."
    )


# ---------------------------------------------------------
# User selection controls
# ---------------------------------------------------------

st.subheader(
    "Select a historical road observation"
)

available_roads = sorted(
    forecasting_data[
        "street_name"
    ].unique()
)

selected_road = st.selectbox(
    "Road",
    options=available_roads
)

road_data = (
    forecasting_data[
        forecasting_data[
            "street_name"
        ] == selected_road
    ]
    .copy()
)

available_dates = sorted(
    road_data[
        "observation_date"
    ].unique()
)

selected_date = st.selectbox(
    "Observation date",
    options=available_dates,
    format_func=lambda value:
    pd.Timestamp(value).strftime(
        "%d %B %Y"
    )
)

date_data = (
    road_data[
        road_data[
            "observation_date"
        ] == selected_date
    ]
    .copy()
)

available_hours = sorted(
    date_data["hour"].astype(int).unique()
)

selected_hour = st.selectbox(
    "Current observation hour",
    options=available_hours,
    format_func=lambda hour:
    f"{int(hour):02d}:00"
)

matching_rows = date_data[
    date_data["hour"].astype(int)
    == int(selected_hour)
]

if matching_rows.empty:
    st.error(
        "No model-ready observation was found for "
        "the selected road, date and hour."
    )

    st.stop()

selected_row = matching_rows.iloc[0]


# ---------------------------------------------------------
# Current observation summary
# ---------------------------------------------------------

st.subheader(
    "Selected observation"
)

summary_column_1, summary_column_2, summary_column_3 = (
    st.columns(3)
)

summary_column_1.metric(
    "Current probe count",
    f'{selected_row["probe_count"]:,.0f}'
)

summary_column_2.metric(
    "Current time",
    selected_row[
        "datetime"
    ].strftime("%d %b %Y, %H:%M")
)

summary_column_3.metric(
    "Prediction target",
    selected_row[
        "target_datetime"
    ].strftime("%d %b %Y, %H:%M")
)


# ---------------------------------------------------------
# Prediction
# ---------------------------------------------------------

if st.button(
    "Predict next-hour probe count",
    type="primary",
    use_container_width=True
):

    model_input = pd.DataFrame(
        [
            selected_row[
                FEATURE_COLUMNS
            ].to_dict()
        ]
    )

    predicted_value = (
        forecasting_model.predict(
            model_input
        )[0]
    )

    actual_value = (
        selected_row[
            "target_next_hour"
        ]
    )

    absolute_error = abs(
        actual_value
        - predicted_value
    )

    percentage_error = (
        absolute_error
        / actual_value
        * 100
        if actual_value != 0
        else np.nan
    )

    st.success(
        "The next-hour forecast was generated successfully."
    )

    result_column_1, result_column_2 = (
        st.columns(2)
    )

    result_column_1.metric(
        "Predicted next-hour probe count",
        f"{predicted_value:,.0f}"
    )

    result_column_2.metric(
        "Recorded next-hour probe count",
        f"{actual_value:,.0f}"
    )

    error_column_1, error_column_2 = (
        st.columns(2)
    )

    error_column_1.metric(
        "Absolute error",
        f"{absolute_error:,.0f}"
    )

    error_column_2.metric(
        "Percentage error",
        f"{percentage_error:.2f}%"
    )

    comparison_data = pd.DataFrame({
        "Measure": [
            "Predicted next-hour probe count",
            "Recorded next-hour probe count"
        ],
        "Probe count": [
            predicted_value,
            actual_value
        ]
    })

    st.subheader(
        "Prediction comparison"
    )

    st.bar_chart(
        comparison_data.set_index(
            "Measure"
        )
    )

    with st.expander(
        "View model-input details"
    ):

        input_display = pd.DataFrame({
            "Feature": FEATURE_COLUMNS,
            "Value": [
                model_input.iloc[0][feature]
                for feature in FEATURE_COLUMNS
            ]
        })

        st.dataframe(
            input_display,
            use_container_width=True,
            hide_index=True
        )


# ---------------------------------------------------------
# Recent road activity
# ---------------------------------------------------------

st.subheader(
    "Recent probe-count pattern"
)

historical_window = (
    road_data[
        road_data["datetime"]
        <= selected_row["datetime"]
    ]
    .tail(24)
    .set_index("datetime")[
        ["probe_count"]
    ]
)

historical_window = historical_window.rename(
    columns={
        "probe_count":
        "Aggregated probe count"
    }
)

st.line_chart(
    historical_window
)


# ---------------------------------------------------------
# Methodological notice
# ---------------------------------------------------------

st.divider()

st.caption(
    "The prototype demonstrates the integration of a saved "
    "machine-learning pipeline with a simple interactive "
    "interface. Results are restricted to the selected roads "
    "and the available 20-day historical dataset. The system "
    "does not currently include live traffic, weather, incident "
    "or seasonal information."
)

Writing app.py


In [11]:
import os

print("STREAMLIT APPLICATION CHECK")
print("=" * 70)

if os.path.exists("app.py"):
    print(
        "[FOUND] app.py",
        f"({os.path.getsize('app.py'):,} bytes)"
    )
else:
    print("[MISSING] app.py")

STREAMLIT APPLICATION CHECK
[FOUND] app.py (10,689 bytes)


In [12]:
!pip -q install streamlit

print("Streamlit installation completed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 91.0 MB/s eta 0:00:00
Streamlit installation completed.


In [13]:
import importlib.metadata as metadata

packages = [
    "streamlit",
    "pandas",
    "numpy",
    "scikit-learn",
    "joblib"
]

print("PACKAGE VERSIONS")
print("=" * 60)

for package in packages:
    print(
        f"{package}:",
        metadata.version(package)
    )

PACKAGE VERSIONS
streamlit: 1.60.0
pandas: 2.2.2
numpy: 2.0.2
scikit-learn: 1.6.1
joblib: 1.5.3


In [14]:
package_lines = [
    f"streamlit=={metadata.version('streamlit')}",
    f"pandas=={metadata.version('pandas')}",
    f"numpy=={metadata.version('numpy')}",
    f"scikit-learn=={metadata.version('scikit-learn')}",
    f"joblib=={metadata.version('joblib')}"
]

with open(
    "requirements.txt",
    "w",
    encoding="utf-8"
) as requirements_file:
    requirements_file.write(
        "\n".join(package_lines)
        + "\n"
    )

print("REQUIREMENTS.TXT")
print("=" * 60)

print(
    open(
        "requirements.txt",
        encoding="utf-8"
    ).read()
)

REQUIREMENTS.TXT
streamlit==1.60.0
pandas==2.2.2
numpy==2.0.2
scikit-learn==1.6.1
joblib==1.5.3



In [15]:
package_lines = [
    f"streamlit=={metadata.version('streamlit')}",
    f"pandas=={metadata.version('pandas')}",
    f"numpy=={metadata.version('numpy')}",
    f"scikit-learn=={metadata.version('scikit-learn')}",
    f"joblib=={metadata.version('joblib')}"
]

with open(
    "requirements.txt",
    "w",
    encoding="utf-8"
) as requirements_file:
    requirements_file.write(
        "\n".join(package_lines)
        + "\n"
    )

print("REQUIREMENTS.TXT")
print("=" * 60)

print(
    open(
        "requirements.txt",
        encoding="utf-8"
    ).read()
)

REQUIREMENTS.TXT
streamlit==1.60.0
pandas==2.2.2
numpy==2.0.2
scikit-learn==1.6.1
joblib==1.5.3



In [16]:
import py_compile

try:
    py_compile.compile(
        "app.py",
        doraise=True
    )

    print(
        "Syntax check passed: app.py"
    )

except py_compile.PyCompileError as error:
    print(
        "Syntax check failed:"
    )

    raise error

Syntax check passed: app.py


In [17]:
import os

application_files = [
    "app.py",
    "requirements.txt",
    "random_forest_next_hour_traffic_model.joblib",
    "selected_delhi_roads_hourly.csv"
]

print("APPLICATION FILE CHECK")
print("=" * 70)

all_application_files_found = True

for filename in application_files:

    if os.path.exists(filename):

        print(
            f"[FOUND] {filename} "
            f"({os.path.getsize(filename):,} bytes)"
        )

    else:

        print(
            f"[MISSING] {filename}"
        )

        all_application_files_found = False

print(
    "\nApplication files ready:",
    all_application_files_found
)

APPLICATION FILE CHECK
[FOUND] app.py (10,689 bytes)
[FOUND] requirements.txt (79 bytes)
[FOUND] random_forest_next_hour_traffic_model.joblib (42,664,954 bytes)
[FOUND] selected_delhi_roads_hourly.csv (376,264 bytes)

Application files ready: True


In [18]:
import subprocess
import time
import requests

streamlit_log = open(
    "streamlit.log",
    "w",
    encoding="utf-8"
)

streamlit_process = subprocess.Popen(
    [
        "python",
        "-m",
        "streamlit",
        "run",
        "app.py",
        "--server.headless=true",
        "--server.port=8501",
        "--server.address=0.0.0.0"
    ],
    stdout=streamlit_log,
    stderr=subprocess.STDOUT
)

print(
    "Streamlit process started with PID:",
    streamlit_process.pid
)

time.sleep(10)

try:

    health_response = requests.get(
        "http://localhost:8501/_stcore/health",
        timeout=10
    )

    print(
        "Health status:",
        health_response.status_code
    )

    print(
        "Health response:",
        health_response.text
    )

except Exception as error:

    print(
        "The Streamlit health check failed."
    )

    print(error)

    print("\nSTREAMLIT LOG")
    print("=" * 70)

    streamlit_log.flush()

    print(
        open(
            "streamlit.log",
            encoding="utf-8"
        ).read()
    )

Streamlit process started with PID: 4876
Health status: 200
Health response: ok


In [19]:
!wget -q \
https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
-O cloudflared

!chmod +x cloudflared

print("Cloudflared downloaded.")

Cloudflared downloaded.


In [20]:
import re
import subprocess
import time

cloudflare_log_path = (
    "cloudflared.log"
)

cloudflare_log = open(
    cloudflare_log_path,
    "w",
    encoding="utf-8"
)

cloudflare_process = subprocess.Popen(
    [
        "./cloudflared",
        "tunnel",
        "--url",
        "http://localhost:8501",
        "--no-autoupdate"
    ],
    stdout=cloudflare_log,
    stderr=subprocess.STDOUT
)

print(
    "Cloudflare tunnel process started with PID:",
    cloudflare_process.pid
)

public_url = None

for attempt in range(40):

    time.sleep(1)

    cloudflare_log.flush()

    with open(
        cloudflare_log_path,
        encoding="utf-8"
    ) as log_file:

        log_text = log_file.read()

    url_match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        log_text
    )

    if url_match:

        public_url = url_match.group(0)
        break

if public_url:

    print("\nSTREAMLIT PUBLIC LINK")
    print("=" * 70)

    print(public_url)

    print(
        "\nOpen this link in a new browser tab."
    )

else:

    print(
        "The public link was not detected."
    )

    print("\nCLOUDFLARE LOG")
    print("=" * 70)

    print(log_text)

Cloudflare tunnel process started with PID: 5035

STREAMLIT PUBLIC LINK
https://peterson-sorts-candles-assistant.trycloudflare.com

Open this link in a new browser tab.


In [21]:
road_prototype_tests = []

for road_name, road_data in prototype_model_df.groupby(
    "street_name"
):
    test_row = (
        road_data
        .sort_values("datetime")
        .iloc[-1]
    )

    test_input = pd.DataFrame([
        test_row[feature_columns]
    ])

    prediction = random_forest_pipeline.predict(
        test_input
    )[0]

    actual = test_row["target_next_hour"]

    absolute_error = abs(
        actual - prediction
    )

    percentage_error = (
        absolute_error / actual * 100
        if actual != 0
        else np.nan
    )

    road_prototype_tests.append({
        "Test_ID": (
            f"PT-{len(road_prototype_tests) + 1:02d}"
        ),
        "Road": road_name,
        "Current_Time": test_row["datetime"],
        "Target_Time": test_row["target_datetime"],
        "Predicted_Probe_Count": prediction,
        "Actual_Probe_Count": actual,
        "Absolute_Error": absolute_error,
        "Percentage_Error": percentage_error,
        "Finite_Prediction": bool(
            np.isfinite(prediction)
        ),
        "Non_Negative_Prediction": bool(
            prediction >= 0
        ),
        "Test_Status": (
            "PASS"
            if np.isfinite(prediction)
            and prediction >= 0
            else "FAIL"
        )
    })

prototype_road_test_df = pd.DataFrame(
    road_prototype_tests
)

print("TEN-ROAD PROTOTYPE FUNCTIONALITY TEST")
print("=" * 80)

display(
    prototype_road_test_df.round(4)
)

print(
    "\nTests passed:",
    (
        prototype_road_test_df["Test_Status"]
        == "PASS"
    ).sum(),
    "out of",
    len(prototype_road_test_df)
)

prototype_road_test_df.to_csv(
    "prototype_ten_road_test_results.csv",
    index=False
)

print(
    "\nSaved: prototype_ten_road_test_results.csv"
)

TEN-ROAD PROTOTYPE FUNCTIONALITY TEST


,Test_ID,Road,Current_Time,Target_Time,Predicted_Probe_Count,Actual_Probe_Count,Absolute_Error,Percentage_Error,Finite_Prediction,Non_Negative_Prediction,Test_Status
0,PT-01,Africa Avenue,2024-08-30 22:00:00,2024-08-30 23:00:00,23188.3361,23311.0,122.6639,0.5262,True,True,PASS
1,PT-02,Aurobindo Marg,2024-08-30 22:00:00,2024-08-30 23:00:00,134198.5796,124796.0,9402.5796,7.5344,True,True,PASS
2,PT-03,Barapullah Road,2024-08-30 22:00:00,2024-08-30 23:00:00,46998.0993,49772.0,2773.9007,5.5732,True,True,PASS
3,PT-04,Mahatma Gandhi Marg,2024-08-30 22:00:00,2024-08-30 23:00:00,399512.9545,361754.0,37758.9545,10.4377,True,True,PASS
4,PT-05,Mathura Road,2024-08-30 22:00:00,2024-08-30 23:00:00,136529.1054,137616.0,1086.8946,0.7898,True,True,PASS
5,PT-06,Mehrauli Badarpur Road,2024-08-30 22:00:00,2024-08-30 23:00:00,53226.6727,54770.0,1543.3273,2.8178,True,True,PASS
6,PT-07,Noida Link Road,2024-08-30 22:00:00,2024-08-30 23:00:00,29106.4234,27888.0,1218.4234,4.3690,True,True,PASS
7,PT-08,Outer Ring Road,2024-08-30 22:00:00,2024-08-30 23:00:00,251755.0649,254266.0,2510.9351,0.9875,True,True,PASS
8,PT-09,Sardar Patel Marg,2024-08-30 22:00:00,2024-08-30 23:00:00,51933.2587,47266.0,4667.2587,9.8745,True,True,PASS
9,PT-10,Vikas Marg,2024-08-30 22:00:00,2024-08-30 23:00:00,44167.3044,44791.0,623.6956,1.3925,True,True,PASS



Tests passed: 10 out of 10

Saved: prototype_ten_road_test_results.csv


In [22]:
from google.colab import files

files.download("app.py")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
files.download("requirements.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
files.download(
    "prototype_readiness_test_result.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
files.download(
    "prototype_ten_road_test_results.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>